IMPORTANT

If you haven't already, create a .venv and install dependencies

After creating a .venv, run this in your code editor's terminal inside the .venv source location: "python -m pip install pandas ipykernel numpy jupyter"

In [192]:
import pandas as pd
import numpy as np

In [193]:
play_attention_scores = pd.read_csv("../outputs_csv/play_attention_scores.csv")
pffScoutingData = pd.read_csv("../cleaned_csv/pffScoutingData_cleaned.csv")
plays = pd.read_csv("../cleaned_csv/plays_cleaned_cleaned.csv")
play_context = pd.read_csv("../outputs_csv/play_context.csv")
# The each_pass_rusher dataframe contains each pass rusher for each play.
each_pass_rusher = pd.read_csv("../outputs_csv/each_pass_rusher.csv")

In [194]:
play_attention_scores.rename(columns={"rusher_nflId": "nflId"}, inplace=True)

**VERY IMPORTANT!**

The base_filtered file below is not in the GitHub because it is too large. It is in a .zip file in the shared **Google Drive** folder inside the "Output Datasets" folder. The zip file is called "Datasets Not in Github (Michael).zip". 

After extracting the .zip file, drag the base_filtered.csv file into the outputs_csv folder.

In [195]:
# The base_filtered dataframe contains frame-by-frame data of all players on only relevant plays.
base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")



/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_17549/3775231977.py:2: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")


In [196]:
base_filtered['is_defense'] = base_filtered['pff_role'].isin(['Coverage', 'Pass Rush'])
base_filtered['team'] = np.where(
    base_filtered['is_defense'],
    base_filtered['defensiveTeam'],
    base_filtered['possessionTeam'],
)

base_filtered[['pff_role', 'possessionTeam', 'defensiveTeam', 'team', 'is_defense']].head()

,pff_role,possessionTeam,defensiveTeam,team,is_defense
0,Pass,TB,DAL,TB,False
1,Pass,TB,DAL,TB,False
2,Pass,TB,DAL,TB,False
3,Pass,TB,DAL,TB,False
4,Pass,TB,DAL,TB,False


In [197]:
pffScoutingData

,gameId,playId,nflId,pff_role,pff_positionLinedUp,pff_nflIdBlockedPlayer,pff_blockType,pff_backFieldBlock
0,2021090900,97,25511,Pass,QB,NaN,NaN,NaN
1,2021090900,97,35481,Pass Route,TE-L,NaN,NaN,NaN
2,2021090900,97,35634,Pass Route,LWR,NaN,NaN,NaN
3,2021090900,97,39985,Pass Route,HB-R,NaN,NaN,NaN
4,2021090900,97,40151,Pass Block,C,44955.0,SW,0.0
...,...,...,...,...,...,...,...,...
188249,2021110100,4433,52507,Pass Block,LT,43338.0,PP,0.0
188250,2021110100,4433,52546,Coverage,SCBoR,NaN,NaN,NaN
188251,2021110100,4433,52573,Pass Route,SLoWR,NaN,NaN,NaN
188252,2021110100,4433,52585,Pass Rush,LEO,NaN,NaN,NaN


Create dataframes that can be used for later.

In [198]:
# ball_snap_frames contains only frames when the ball is snapped
ball_snap_frames = base_filtered[base_filtered['event'] == 'ball_snap']

# base_filtered_pass_rushers is for pass rushers only, and it includes frames before the pass rush window unlike pass_rushers.
base_filtered_pass_rushers = base_filtered.merge(each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates(), on=['gameId', 'playId', 'nflId'], how='inner')

Information for LLM or AI prompting: Each combination of gameId and playId represents an individual play. Each combination of gameId, playId, and nflId represents an individual player on an indiviual play.

In [199]:
ball_snap_frames.columns

Index(['gameId', 'playId', 'season', 'week', 'gameDate', 'quarter', 'down',
       'yardsToGo', 'gameClock', 'play time', 'frameId', 'possessionTeam',
       'defensiveTeam', 'yardlineSide', 'yardlineNumber',
       'absoluteYardlineNumber', 'offenseFormation', 'offenseRB', 'offenseTE',
       'offenseWR', 'defendersInBox', 'defenseDL', 'defenseLB', 'defenseDB',
       'dropBackType', 'playAction', 'passCoverage', 'passCoverageType',
       'is_screen', 'is_rpo', 'is_qb_spike', 'is_gravity_candidate_base',
       'preSnapHomeScore', 'preSnapVisitorScore', 'passResult', 'penaltyYards',
       'prePenaltyPlayResult', 'playResult', 'foulNames', 'foulIds', 'nflId',
       'pff_role', 'pff_positionLinedUp', 'pff_nflIdBlockedPlayer',
       'pff_blockType', 'pff_backFieldBlock', 'height', 'weight',
       'officialPosition', 'displayName', 'jerseyNumber', 'playDirection', 'x',
       'y', 's', 'a', 'dis', 'o', 'dir', 'event', 'frameIdEndWindow',
       'is_defense', 'team'],
      dtype='obj

Feature-engineer variables for how far the pass rusher is from the center at the ball_snap instance (x distance, y distance, and total distance).

In [200]:


# filter ball_snap_frames to only include ball snap frames of pass rushers in each_pass_rusher df
ball_snaps_pass_rushers = ball_snap_frames.merge(each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates(), on=['gameId', 'playId', 'nflId'], how='inner')
ball_snaps_pass_rushers

# ball_snap_frames to only include centers
ball_snaps_centers = ball_snap_frames[ball_snap_frames['pff_positionLinedUp'] == 'C']
ball_snaps_centers

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,y,s,a,dis,o,dir,event,frameIdEndWindow,is_defense,team
149,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,24.02,0.51,1.98,0.06,48.28,308.34,ball_snap,36,False,TB
1418,2021090900,137,2021,1,09/09/2021,1,1,10,13:18,28:10.500,...,24.06,0.92,0.88,0.09,279.04,170.92,ball_snap,31,False,DAL
2046,2021090900,187,2021,1,09/09/2021,1,2,6,12:23,29:15.500,...,26.46,0.54,1.41,0.06,284.80,121.34,ball_snap,27,False,DAL
2829,2021090900,282,2021,1,09/09/2021,1,1,10,9:56,31:52.100,...,30.07,0.58,0.87,0.06,265.80,123.15,ball_snap,36,False,DAL
3506,2021090900,349,2021,1,09/09/2021,1,3,15,9:46,34:05.600,...,30.11,0.82,2.07,0.08,276.68,94.61,ball_snap,32,False,DAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5163669,2021110100,4310,2021,8,11/01/2021,4,3,8,1:56,15:52.900,...,30.05,0.39,1.38,0.03,282.30,91.81,ball_snap,37,False,KC
5164113,2021110100,4363,2021,8,11/01/2021,4,1,10,1:07,18:41.000,...,29.92,0.55,1.39,0.05,75.80,288.03,ball_snap,37,False,NYG
5164927,2021110100,4392,2021,8,11/01/2021,4,2,7,1:01,19:17.900,...,23.77,0.54,1.54,0.06,79.79,278.44,ball_snap,37,False,NYG
5165696,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:40.200,...,23.78,0.62,1.08,0.06,81.31,273.62,ball_snap,32,False,NYG


In [201]:
# calculate the difference in x, the difference in y, and the euclidean distance of each pass rusher to the center at the moment of the ball snap on each play.
temp = ball_snaps_pass_rushers.merge(ball_snaps_centers[['gameId', 'playId', 'x', 'y']], on=['gameId', 'playId'], how='inner', suffixes=('', '_center'))
temp['diff_x'] = temp['x'] - temp['x_center']
temp['diff_y'] = temp['y'] - temp['y_center']
temp['euclidean_distance'] = np.sqrt(temp['diff_x']**2 + temp['diff_y']**2)
temp

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,dir,event,frameIdEndWindow,is_defense,team,x_center,y_center,diff_x,diff_y,euclidean_distance
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,288.76,ball_snap,36,True,DAL,42.10,24.02,1.20,-5.13,5.268482
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,247.75,ball_snap,36,True,DAL,42.10,24.02,1.80,8.61,8.796141
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,288.42,ball_snap,36,True,DAL,42.10,24.02,1.25,1.16,1.705315
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,316.78,ball_snap,36,True,DAL,42.10,24.02,1.58,-2.09,2.620019
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,345.84,ball_snap,36,True,DAL,42.10,24.02,1.60,2.65,3.095561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:40.200,...,249.67,ball_snap,32,True,KC,29.07,23.78,1.22,4.86,5.010788
30967,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,271.87,ball_snap,37,True,KC,29.15,23.72,1.16,6.07,6.179846
30968,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,251.87,ball_snap,37,True,KC,29.15,23.72,1.04,-2.90,3.080844
30969,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,283.80,ball_snap,37,True,KC,29.15,23.72,1.30,2.93,3.205448


In [202]:
# create a training dataset for each pass rusher on each play.
training_dataset = each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates()
training_dataset = training_dataset.merge(temp[['gameId', 'playId', 'nflId', 'diff_x', 'diff_y', 'euclidean_distance']], on=['gameId', 'playId', 'nflId'], how='left')




Feature-engineer other variables and add them to the `training_dataset` dataframe.

PFF Position Lined Up

In [203]:
temp = ball_snaps_pass_rushers.copy()
temp['pff_positionLinedUp'].value_counts()


# group RE and LE. group REO and LEO. group ROLB and LOLB. group DRT and DLT. group NLT, NRT, and NT.
def group_positions(position):
    if position in ['RE', 'LE']:
        return 'End'
    elif position in ['REO', 'LEO']:
        return 'End Outside'
    elif position in ['ROLB', 'LOLB']:
        return 'OLB'
    elif position in ['DRT', 'DLT']:
        return 'DT'
    elif position in ['NLT', 'NRT', 'NT']:
        return 'NT'
    elif position in ['RILB', 'LILB', 'MLB']:
        return 'ILB'
    elif position in ['RLB', 'LLB']:
        return 'LB'
    else:
        return "Secondary"
    
temp['grouped_position'] = temp['pff_positionLinedUp'].apply(group_positions)

# merge play_attention_scores with ball_snaps_pass_rushers on gameId, playId, and nflId to get the grouped for each pass rusher in the play_attention_scores dataframe.
scores_temp = play_attention_scores.merge(
    temp[['gameId', 'playId', 'nflId', 'grouped_position']],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)
scores_temp.groupby('grouped_position')['avg_attention_score'].mean().sort_values(ascending=False)

grouped_position
DT             1.440668
End            1.426816
NT             1.423070
End Outside    1.008230
OLB            0.885306
ILB            0.864821
LB             0.541355
Secondary      0.309285
Name: avg_attention_score, dtype: float64

In [204]:
# group "DT", "NT", and "End" as "DI". group "End Outside" and "OLB" as "Edge". group "LB" and "Secondary" as "Other"

# create function
def group_positions_final(position):
    if position in ['DT', 'NT', 'End']:
        return 'DI'
    elif position in ['End Outside', 'OLB']:
        return 'Edge'
    elif position in ['LB', 'Secondary']:
        return 'Other'
    else:
        return position

temp['grouped_position'] = temp['grouped_position'].apply(group_positions_final)

# do one-hot encoding on temp['grouped_position'] column and merge it back to the training_dataset on gameId, playId, and nflId.

one_hot = pd.get_dummies(temp['grouped_position'], prefix='pos')
temp = pd.concat([temp[['gameId', 'playId', 'nflId']], one_hot], axis=1)
training_dataset = training_dataset.merge(
    temp,
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)




Number of rushers who are rushing

In [205]:
# Count pass rushers by play. playId can repeat across games, so group by both gameId and playId.
num_blitzing_rushers_by_play = (
    each_pass_rusher[['gameId', 'playId', 'nflId']]
    .drop_duplicates()
    .groupby(['gameId', 'playId'], as_index=False)
    .agg(num_blitzing_rushers=('nflId', 'count'))
)

training_dataset = training_dataset.merge(
    num_blitzing_rushers_by_play,
    on=['gameId', 'playId'],
    how='left',
    validate='many_to_one',
)

training_dataset

,gameId,playId,nflId,diff_x,diff_y,euclidean_distance,pos_DI,pos_Edge,pos_ILB,pos_Other,num_blitzing_rushers
0,2021090900,97,41263,1.20,-5.13,5.268482,0,1,0,0,5
1,2021090900,97,42403,1.80,8.61,8.796141,0,1,0,0,5
2,2021090900,97,44955,1.25,1.16,1.705315,1,0,0,0,5
3,2021090900,97,53441,1.58,-2.09,2.620019,0,0,1,0,5
4,2021090900,97,53504,1.60,2.65,3.095561,1,0,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,52585,1.22,4.86,5.010788,0,1,0,0,5
30967,2021110100,4433,42406,1.16,6.07,6.179846,0,1,0,0,4
30968,2021110100,4433,43326,1.04,-2.90,3.080844,1,0,0,0,4
30969,2021110100,4433,43338,1.30,2.93,3.205448,1,0,0,0,4


Number of "expected rushers" - defensive players lined up near the LOS (excluding CBs)

In [206]:
## number of defensive players lined up within 2.5 yards in x distance from the center. Do not include any type of cornerbacks.

center_locations = (
    ball_snaps_centers[['gameId', 'playId', 'x']]
    .drop_duplicates(subset=['gameId', 'playId'])
    .rename(columns={'x': 'x_center'})
)

ball_snap_defenders = ball_snap_frames.merge(
    center_locations,
    on=['gameId', 'playId'],
    how='inner',
    validate='many_to_one',
)

non_cornerback_defender_near_center = (
    (ball_snap_defenders['team'] == ball_snap_defenders['defensiveTeam'])
    & ((ball_snap_defenders['x'] - ball_snap_defenders['x_center']).abs() <= 2.5)
    & (~ball_snap_defenders['pff_positionLinedUp'].str.contains('CB', case=False, na=False))
)

defenders_on_los = (
    ball_snap_defenders[non_cornerback_defender_near_center]
    .groupby(['gameId', 'playId'], as_index=False)
    .agg(numExpectedRushers=('nflId', 'nunique'))
)

training_dataset = training_dataset.merge(
    defenders_on_los,
    on=['gameId', 'playId'],
    how='left',
    validate='many_to_one',
)

training_dataset['numExpectedRushers'] = training_dataset['numExpectedRushers'].fillna(0).astype(int)

# blitzers vs expected = number of blitzing rushers - number of defenders near the line of scrimmage
training_dataset['blitzers_vs_expected'] = training_dataset['num_blitzing_rushers'] - training_dataset['numExpectedRushers']

In [207]:
# merge training_dataset with the above "ball_snap_defenders[non_cornerback_defender_near_center]" to get a boolean feature to see if the pass rusher is lined up on the line of scrimmage. Use gameId, playId, and nflId. If not, then it is false.
pass_rushers_on_los = (
    ball_snap_defenders.loc[
        non_cornerback_defender_near_center,
        ['gameId', 'playId', 'nflId'],
    ]
    .drop_duplicates()
    .assign(isExpectedRusher=True)
)

training_dataset = training_dataset.merge(
    pass_rushers_on_los,
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)

training_dataset['isExpectedRusher'] = training_dataset['isExpectedRusher'].fillna(False)

# convert above column to 0s and 1s
training_dataset['isExpectedRusher'] = training_dataset['isExpectedRusher'].astype(int)

In [208]:
training_dataset['isExpectedRusher'].value_counts()

1    29492
0     1479
Name: isExpectedRusher, dtype: int64

Number of rushers on the play who weren't "expected rushers"

In [209]:
# Count pass rushers by play who rushed from away from the line of scrimmage.
# Cornerbacks who rush are counted as not near the LOS, matching pass_rusher_near_los.
pass_rusher_not_near_los = training_dataset['isExpectedRusher'] == 0

non_los_pass_rushers_by_play = (
    training_dataset.loc[pass_rusher_not_near_los, ['gameId', 'playId', 'nflId']]
    .drop_duplicates()
    .groupby(['gameId', 'playId'], as_index=False)
    .agg(num_non_expected_rushers=('nflId', 'count'))
)

training_dataset = training_dataset.merge(
    non_los_pass_rushers_by_play,
    on=['gameId', 'playId'],
    how='left',
    validate='many_to_one',
)

training_dataset['num_non_expected_rushers'] = training_dataset['num_non_expected_rushers'].fillna(0).astype(int)
training_dataset['has_non_expected_rusher'] = (training_dataset['num_non_expected_rushers'] > 0).astype(int)
training_dataset['potential_rushers'] = training_dataset['numExpectedRushers'] + training_dataset['num_non_expected_rushers']


Number of pass rushers who are on the rusher's side of the center

In [210]:
# calculate the number and proportion of other pass rushers who are on each rusher's side of the center at the moment of the ball snap on each play. 
pass_rusher_sides = training_dataset[['gameId', 'playId', 'nflId', 'diff_y', 'num_blitzing_rushers']].drop_duplicates().copy()
pass_rusher_sides['rusher_side_of_center'] = np.select(
    [pass_rusher_sides['diff_y'] > 0, pass_rusher_sides['diff_y'] < 0],
    [1, -1],
    default=0,
)

pass_rushers_by_side = (
    pass_rusher_sides
    .groupby(['gameId', 'playId', 'rusher_side_of_center'], as_index=False)
    .agg(numPassRushersOnRusherSide=('nflId', 'nunique'))
)

other_pass_rusher_side_features = pass_rusher_sides.merge(
    pass_rushers_by_side,
    on=['gameId', 'playId', 'rusher_side_of_center'],
    how='left',
    validate='many_to_one',
)

other_pass_rusher_side_features['numOtherPassRushersOnRusherSide'] = (
    other_pass_rusher_side_features['numPassRushersOnRusherSide'] - 1
).clip(lower=0).astype(int)

other_pass_rushers_by_play = other_pass_rusher_side_features['num_blitzing_rushers'] - 1
other_pass_rusher_side_features['propOtherPassRushersOnRusherSide'] = np.where(
    other_pass_rushers_by_play > 0,
    other_pass_rusher_side_features['numOtherPassRushersOnRusherSide'] / other_pass_rushers_by_play,
    0,
)

training_dataset = training_dataset.merge(
    other_pass_rusher_side_features[
        [
            'gameId',
            'playId',
            'nflId',
            'numOtherPassRushersOnRusherSide',
            'propOtherPassRushersOnRusherSide',
        ]
    ],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)

training_dataset['numOtherPassRushersOnRusherSide'] = training_dataset['numOtherPassRushersOnRusherSide'].fillna(0).astype(int)
training_dataset['propOtherPassRushersOnRusherSide'] = training_dataset['propOtherPassRushersOnRusherSide'].fillna(0)


In [211]:
# calculate the number and proportion of expected pass rushers (players lined up near the LOS and not a CB) who are on the rusher's side of the center.
center_y_locations = (
    ball_snaps_centers[['gameId', 'playId', 'y']]
    .drop_duplicates(subset=['gameId', 'playId'])
    .rename(columns={'y': 'y_center'})
)

expected_rusher_sides = (
    ball_snap_defenders.loc[
        non_cornerback_defender_near_center,
        ['gameId', 'playId', 'nflId', 'y'],
    ]
    .drop_duplicates()
    .merge(
        center_y_locations,
        on=['gameId', 'playId'],
        how='inner',
        validate='many_to_one',
    )
)

expected_rusher_sides['rusher_side_of_center'] = np.select(
    [expected_rusher_sides['y'] > expected_rusher_sides['y_center'], expected_rusher_sides['y'] < expected_rusher_sides['y_center']],
    [1, -1],
    default=0,
)

expected_rushers_by_side = (
    expected_rusher_sides
    .groupby(['gameId', 'playId', 'rusher_side_of_center'], as_index=False)
    .agg(numExpectedRushersOnRusherSide=('nflId', 'nunique'))
)

pass_rusher_sides = training_dataset[['gameId', 'playId', 'nflId', 'diff_y', 'numExpectedRushers', 'isExpectedRusher']].drop_duplicates().copy()
pass_rusher_sides['rusher_side_of_center'] = np.select(
    [pass_rusher_sides['diff_y'] > 0, pass_rusher_sides['diff_y'] < 0],
    [1, -1],
    default=0,
)

expected_rusher_side_features = pass_rusher_sides.merge(
    expected_rushers_by_side,
    on=['gameId', 'playId', 'rusher_side_of_center'],
    how='left',
    validate='many_to_one',
)

expected_rusher_side_features['numExpectedRushersOnRusherSide'] = expected_rusher_side_features['numExpectedRushersOnRusherSide'].fillna(0).astype(int)
other_expected_rushers_on_side = (
    expected_rusher_side_features['numExpectedRushersOnRusherSide'] - expected_rusher_side_features['isExpectedRusher']
).clip(lower=0)
other_expected_rushers_by_play = (
    expected_rusher_side_features['numExpectedRushers'] - expected_rusher_side_features['isExpectedRusher']
).clip(lower=0)
expected_rusher_side_features['numExpectedRushersOnRusherSide'] = other_expected_rushers_on_side.astype(int)

expected_rusher_side_features['propExpectedRushersOnRusherSide'] = np.where(
    other_expected_rushers_by_play > 0,
    other_expected_rushers_on_side / other_expected_rushers_by_play,
    0,
)

training_dataset = training_dataset.merge(
    expected_rusher_side_features[
        [
            'gameId',
            'playId',
            'nflId',
            'numExpectedRushersOnRusherSide',
            'propExpectedRushersOnRusherSide',
        ]
    ],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)

training_dataset['numExpectedRushersOnRusherSide'] = training_dataset['numExpectedRushersOnRusherSide'].fillna(0).astype(int)
training_dataset['propExpectedRushersOnRusherSide'] = training_dataset['propExpectedRushersOnRusherSide'].fillna(0)

In [212]:
# # test for correctness

# temp = ball_snap_defenders.loc[non_cornerback_defender_near_center,
#         ['gameId', 'playId', 'nflId', 'y'],
#     ].drop_duplicates().merge(
#         center_y_locations,
#         on=['gameId', 'playId'],
#         how='inner',
#         validate='many_to_one',
#     )
# print(temp[(temp['gameId'] == 2021090900) & (temp['playId'] == 1563) ])

# training_dataset[(training_dataset['gameId'] == 2021090900) & (training_dataset['playId'] == 1563) ]


Down variable: Feature-engineer down variable using one-hot encoding. We group downs 3 and 4

In [213]:
ball_snaps_pass_rushers['down_grouped'] = ball_snaps_pass_rushers['down'].apply(lambda x: '3-4' if x in [3, 4] else str(x))

# do one-hot encoding for down using ball_snaps_pass_rushers['down'] and merge to training_dataset. Use "down_grouped".  
down_dummies = pd.get_dummies(ball_snaps_pass_rushers['down_grouped'], prefix='down')
ball_snaps_pass_rushers_with_dummies = pd.concat([ball_snaps_pass_rushers[['gameId', 'playId', 'nflId']], down_dummies], axis=1)
training_dataset = training_dataset.merge(
    ball_snaps_pass_rushers_with_dummies,
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)   

In [219]:
training_dataset.iloc[:,0:20]

,gameId,playId,nflId,diff_x,diff_y,euclidean_distance,pos_DI,pos_Edge,pos_ILB,pos_Other,num_blitzing_rushers,numExpectedRushers,blitzers_vs_expected,isExpectedRusher,num_non_expected_rushers,has_non_expected_rusher,potential_rushers,numOtherPassRushersOnRusherSide,propOtherPassRushersOnRusherSide,numExpectedRushersOnRusherSide
0,2021090900,97,41263,1.20,-5.13,5.268482,0,1,0,0,5,5,0,1,0,0,5,1,0.250000,1
1,2021090900,97,42403,1.80,8.61,8.796141,0,1,0,0,5,5,0,1,0,0,5,2,0.500000,2
2,2021090900,97,44955,1.25,1.16,1.705315,1,0,0,0,5,5,0,1,0,0,5,2,0.500000,2
3,2021090900,97,53441,1.58,-2.09,2.620019,0,0,1,0,5,5,0,1,0,0,5,1,0.250000,1
4,2021090900,97,53504,1.60,2.65,3.095561,1,0,0,0,5,5,0,1,0,0,5,2,0.500000,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,52585,1.22,4.86,5.010788,0,1,0,0,5,4,1,1,1,1,5,1,0.250000,1
30967,2021110100,4433,42406,1.16,6.07,6.179846,0,1,0,0,4,5,-1,1,0,0,5,1,0.333333,1
30968,2021110100,4433,43326,1.04,-2.90,3.080844,1,0,0,0,4,5,-1,1,0,0,5,1,0.333333,2
30969,2021110100,4433,43338,1.30,2.93,3.205448,1,0,0,0,4,5,-1,1,0,0,5,1,0.333333,1


Add additional features.

In [215]:
# add more features

### Constructing the Model

In [216]:
training_dataset = training_dataset.merge(play_attention_scores, on=['gameId', 'playId', 'nflId'], how='inner')
training_dataset


,gameId,playId,nflId,diff_x,diff_y,euclidean_distance,pos_DI,pos_Edge,pos_ILB,pos_Other,...,potential_rushers,numOtherPassRushersOnRusherSide,propOtherPassRushersOnRusherSide,numExpectedRushersOnRusherSide,propExpectedRushersOnRusherSide,down_1,down_2,down_3-4,avg_attention_score,rusher_name
0,2021090900,97,41263,1.20,-5.13,5.268482,0,1,0,0,...,5,1,0.250000,1,0.250000,0,0,1,0.461538,Demarcus Lawrence
1,2021090900,97,42403,1.80,8.61,8.796141,0,1,0,0,...,5,2,0.500000,2,0.500000,0,0,1,0.615385,Randy Gregory
2,2021090900,97,44955,1.25,1.16,1.705315,1,0,0,0,...,5,2,0.500000,2,0.500000,0,0,1,1.307692,Carlos Watkins
3,2021090900,97,53441,1.58,-2.09,2.620019,0,0,1,0,...,5,1,0.250000,1,0.250000,0,0,1,1.000000,Micah Parsons
4,2021090900,97,53504,1.60,2.65,3.095561,1,0,0,0,...,5,2,0.500000,2,0.500000,0,0,1,0.615385,Osa Odighizuwa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,52585,1.22,4.86,5.010788,0,1,0,0,...,5,1,0.250000,1,0.333333,0,0,1,0.904762,Michael Danna
30967,2021110100,4433,42406,1.16,6.07,6.179846,0,1,0,0,...,5,1,0.333333,1,0.250000,0,0,1,0.923077,Frank Clark
30968,2021110100,4433,43326,1.04,-2.90,3.080844,1,0,0,0,...,5,1,0.333333,2,0.500000,0,0,1,1.500000,Chris Jones
30969,2021110100,4433,43338,1.30,2.93,3.205448,1,0,0,0,...,5,1,0.333333,1,0.250000,0,0,1,1.346154,Jarran Reed


In [217]:
# # let x be all features except first three and last two columns
# X = training_dataset.drop(columns=['gameId', 'playId', 'nflId', 'avg_attention_score', 'rusher_name'])
# y = training_dataset['avg_attention_score']


# from sklearn.linear_model import LinearRegression
# model = LinearRegression()
# model.fit(X, y)
# print("Coefficients:", model.coef_)
# print("Intercept:", model.intercept_)

# # r2score
# from sklearn.metrics import r2_score
# y_pred = model.predict(X)
# print("R^2 Score:", r2_score(y, y_pred))

Preliminary multiple regression test (Not Our Final Model)

In [218]:
# # Calculate gravity for Myles Garrett based on this regression model.
# myles_garrett_temp = training_dataset[training_dataset['displayName'] == 'Myles Garrett']
# myles_garrett_temp['predicted_attention_score'] = model.predict(myles_garrett_temp[['euclidean_distance']])
# myles_garrett_temp['gravity_score'] = myles_garrett_temp['avg_attention_score'] - myles_garrett_temp['predicted_attention_score']
# myles_garrett_avg_gravity = myles_garrett_temp['gravity_score'].mean()
# print("Myles Garrett's average gravity:", myles_garrett_avg_gravity)  

# # Calculate gravity for Aaron Donald based on this regression model.
# aaron_donald_temp = training_dataset[training_dataset['displayName'] == 'Aaron Donald']
# aaron_donald_temp['predicted_attention_score'] = model.predict(aaron_donald_temp[['euclidean_distance']])
# aaron_donald_temp['gravity_score'] = aaron_donald_temp['avg_attention_score'] - aaron_donald_temp['predicted_attention_score']
# aaron_donald_avg_gravity = aaron_donald_temp['gravity_score'].mean()
# print("Aaron Donald's average gravity:", aaron_donald_avg_gravity)  

# # Calculate gravity for Justin Hollins based on this regression model.
# justin_hollins_temp = training_dataset[training_dataset['displayName'] == 'Justin Hollins']
# justin_hollins_temp['predicted_attention_score'] = model.predict(justin_hollins_temp[['euclidean_distance']])
# justin_hollins_temp['gravity_score'] = justin_hollins_temp['avg_attention_score'] - justin_hollins_temp['predicted_attention_score']
# justin_hollins_avg_gravity = justin_hollins_temp['gravity_score'].mean()
# print("Justin Hollins's average gravity:", justin_hollins_avg_gravity)
